# Step 47 — Hypocentre source comparison

Compares the five hypocentre sources after their clock conversion (step 40) and grouping (step 43). It reads and writes nothing else in the pipeline; its tables and figures support the paper's provenance section and check the clock rules.

| Source | File | Program, processing date |
|---|---|---|
| `hypo40` | `Pinatubo_all.sum` (= `moriall.sum`) | HYPO71, 1991 |
| `pinaall` | `PINAALL.DAT` (= `PINAHYPO.DAT`) | HYPO71, 1991 |
| `gold` | `GOLD.SUM` | HYPOELLIPSE, July 1992 to April 1993 |
| `brun` | `b-run.sum` | HYPOELLIPSE, May 2001 |
| `pepi` | `PinatuboRelocs.rtf` | high-precision locations published by Jones, Thurber and Lutter (2001), *Phys. Earth Planet. Inter.* 123, 221–232 |

**Outputs** (in `RELOC_QC_DIR`):
- `47_source_inventory.csv`: records, time span, depth and magnitude per source, with lines rejected in step 40
- `47_clock_check.csv`: for each source and clock rule, the fraction of origins within 2 s of a HYPO71 origin at the rule's offset and at ±7 h and ±8 h from it. The rule is supported when only the rule's offset matches.
- `47_pair_statistics.csv`: time, epicentre, depth and magnitude differences between sources grouped into the same event in step 43
- `47_pair_differences.png`, `47_pair_maps.png`, `47_gold_processing_dates.png`

Magnitudes differ systematically between sources, so step 43 prefers only HYPO71 magnitudes; the medians here are the evidence.

## Configuration

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'lib' / 'clock.py').is_file()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from lib.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
OUT_DIR = Path(P['RELOC_QC_DIR'])
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEXES = {'hypo40': P['SUMMARY_FILE_40_INDEX'], 'pinaall': P['SUMMARY_FILE_41_INDEX'],
           'gold': P['GOLD_INDEX'], 'brun': P['BRUN_INDEX'], 'pepi': P['PEPI_INDEX']}
REJECTS = {'gold': P['GOLD_ERR'], 'brun': P['BRUN_ERR'], 'pepi': P['PEPI_ERR']}
ORIGIN_INDEX = P['HYPO71_ORIGIN_INDEX']   # step 43 output (all sources)
EVENT_INDEX = P['HYPO71_EVENT_INDEX']
MATCH_S = 2.0                               # clock check: origin within 2 s of a HYPO71 origin
PAIRS = [('pinaall', 'hypo40'), ('hypo40', 'brun'), ('gold', 'brun'), ('gold', 'hypo40'),
         ('pepi', 'brun'), ('pepi', 'hypo40')]
print('Outputs:', OUT_DIR)

## 1. Inventory

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

idx = {s: pd.read_csv(path) for s, path in INDEXES.items()}
for frame in idx.values():
    frame['t'] = pd.to_datetime(frame['origin_time'], format='ISO8601', utc=True)
rows = []
for s, frame in idx.items():
    rejected = pd.read_csv(REJECTS[s]).reason.value_counts().to_dict() if s in REJECTS else {}
    unresolved = Path(INDEXES[s]).with_name(Path(INDEXES[s]).stem + '_clock_unresolved.csv')
    rows.append(dict(source=s, file=frame.source_file.iloc[0], origins=len(frame),
                     first=frame.t.min().strftime('%Y-%m-%d'), last=frame.t.max().strftime('%Y-%m-%d'),
                     depth_median_km=round(frame.depth_km.median(), 2),
                     magnitude_median=round(frame.magnitude.median(), 2),
                     magnitudes=int(frame.magnitude.notna().sum()),
                     clock_unresolved=len(pd.read_csv(unresolved)) if unresolved.exists() else 0,
                     rejected_no_location=rejected.get('no_location', 0),
                     rejected_other=sum(v for k, v in rejected.items() if k not in ('no_location', 'blank'))))
inventory = pd.DataFrame(rows)
inventory.to_csv(OUT_DIR / '47_source_inventory.csv', index=False)
display(inventory)

## 2. Clock check

HYPO71 origins are the reference (W clock, converted in step 40). A rule offset is supported when its origins match HYPO71 at the rule's offset and not at the alternatives. HYPO71 covers fewer events than BRUN, so the fractions are below 1 even when the rule is right.

In [ ]:
ref = np.sort(pd.concat([idx['hypo40'].t, idx['pinaall'].t]).values)

def matched_fraction(times):
    pos = np.searchsorted(ref, times)
    gap = np.minimum(np.abs(ref[np.clip(pos, 0, len(ref) - 1)] - times),
                     np.abs(ref[np.clip(pos - 1, 0, len(ref) - 1)] - times)) / np.timedelta64(1, 's')
    return float((gap <= MATCH_S).mean())

SHIFTS_H = (-8, -7, 0, 7, 8)
rows = []
for s in ('gold', 'brun', 'pepi'):
    frame = idx[s]
    for regime, group in frame.groupby(frame.clock_regime.str.replace('_matched', '', regex=False)):
        t = group.t.dt.tz_localize(None).values
        rows.append(dict(source=s, regime=regime, offset_h=group.clock_offset_s.iloc[0] / 3600, origins=len(group),
                         **{f'match_{h:+d}h': round(matched_fraction(t + np.timedelta64(h, 'h')), 3) for h in SHIFTS_H}))
clock_check = pd.DataFrame(rows)
clock_check['rule_supported'] = (clock_check['match_+0h'] > 0.2) & (
    clock_check[[f'match_{h:+d}h' for h in SHIFTS_H if h]].max(axis=1) < 0.02)
clock_check.to_csv(OUT_DIR / '47_clock_check.csv', index=False)
display(clock_check)
if not clock_check.rule_supported.all():
    print('WARNING: some clock rules are not supported by HYPO71 matching; review before step 50.')

## 3. Differences between sources in the same event

All historical origin pairs are retained. Statistics and histograms are **solution-pair weighted**, not equal-weight event estimates. Tables report unique event groups and origin pairs separately; per-pair CSVs retain both origin IDs for review. Selecting one scientific representative per source remains a separate decision.

Pairs come from Step 43's anchor-based groups: non-anchor members need not be within the distance tolerance of each other. Differences are first minus second source. Histogram limits hide outliers without moving them into endpoint bins; medians use all finite observations.


In [ ]:
origins = pd.read_csv(ORIGIN_INDEX)
origins['t'] = pd.to_datetime(origins['origin_time'], format='ISO8601', utc=True)
by_source = {s: g.set_index('event_id') for s, g in origins.groupby('source')}

def pair_table(a, b):
    if a not in by_source or b not in by_source:
        return pd.DataFrame()
    j = by_source[a].join(by_source[b], how='inner', lsuffix='_a', rsuffix='_b')
    dlat = (j.latitude_a - j.latitude_b) * 111.19
    dlon = (j.longitude_a - j.longitude_b) * 111.19 * np.cos(np.radians(j.latitude_b))
    return pd.DataFrame(dict(origin_id_a=j.origin_id_a, origin_id_b=j.origin_id_b, dt_s=(j.t_a - j.t_b).dt.total_seconds(), dist_km=np.hypot(dlat, dlon),
                             ddepth_km=j.depth_km_a - j.depth_km_b, dmag=j.magnitude_a - j.magnitude_b,
                             lat_a=j.latitude_a, lon_a=j.longitude_a, lat_b=j.latitude_b, lon_b=j.longitude_b))

pairs = {(a, b): pair_table(a, b) for a, b in PAIRS}
for (a, b), table in pairs.items():
    table.rename_axis('event_id').to_csv(OUT_DIR / f'47_pairs_{a}_{b}.csv')
rows = []
for (a, b), d in pairs.items():
    if d.empty:
        continue
    m = d.dmag.dropna()
    rows.append(dict(first=a, second=b, events=d.index.nunique(), origin_pairs=len(d), statistics_unit="origin_pair", dt_median_s=round(d.dt_s.median(), 2), dt_std_s=round(d.dt_s.std(), 2),
                     dist_median_km=round(d.dist_km.median(), 2), dist_p90_km=round(d.dist_km.quantile(0.9), 2),
                     ddepth_median_km=round(d.ddepth_km.median(), 2), ddepth_std_km=round(d.ddepth_km.std(), 2),
                     dmag_median=round(m.median(), 2) if len(m) else None, dmag_std=round(m.std(), 2) if len(m) else None,
                     magnitude_pairs=len(m)))
pair_stats = pd.DataFrame(rows)
pair_stats.to_csv(OUT_DIR / '47_pair_statistics.csv', index=False)
display(pair_stats)

In [ ]:
shown = [k for k, d in pairs.items() if not d.empty]
fig, axes = plt.subplots(len(shown), 3, figsize=(12, 2.4 * len(shown)), squeeze=False)
for row, (a, b) in zip(axes, shown):
    d = pairs[(a, b)]
    for ax, col, label, lim in zip(row, ('dist_km', 'ddepth_km', 'dmag'),
                                   ('epicentre separation (km)', 'depth difference (km)', 'magnitude difference'),
                                   ((0, 10), (-10, 10), (-3, 3))):
        values = d[col].dropna()
        ax.hist(values, bins=40, range=lim, color='0.5')
        if len(values):
            ax.axvline(values.median(), color='k', lw=1)
        ax.set_xlabel(f'{a} − {b}: {label}' if col != 'dist_km' else f'{a} vs {b}: {label}')
        ax.set_ylabel('origin pairs')
        outside = int(((values < lim[0]) | (values > lim[1])).sum())
        ax.text(.98, .95, f'{outside} outside range', transform=ax.transAxes, ha='right', va='top', fontsize=7)
        ax.set_xlim(*lim)
fig.tight_layout()
fig.savefig(OUT_DIR / '47_pair_differences.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, len(shown), figsize=(3.2 * len(shown), 3.6), sharey=True, squeeze=False)
for ax, (a, b) in zip(axes[0], shown):
    d = pairs[(a, b)]
    ax.plot(d.lon_b, d.lat_b, '.', ms=1.5, color='0.6', label=b)
    ax.plot(d.lon_a, d.lat_a, '.', ms=1.5, color='k', label=a)
    ax.plot(120.35, 15.14, '^', color='C3', ms=8)
    ax.set_xlim(120.1, 120.65); ax.set_ylim(14.9, 15.5); ax.set_aspect(1 / np.cos(np.radians(15.14)))
    ax.set_title(f'{a} vs {b} ({d.index.nunique()} groups; {len(d)} pairs)', fontsize=9); ax.legend(fontsize=7, markerscale=5, loc='lower left')
axes[0][0].set_ylabel('latitude (°N)')
fig.tight_layout()
fig.savefig(OUT_DIR / '47_pair_maps.png', dpi=150)
plt.show()

## 4. Processing dates and preferred sources

GOLD records carry the date each was processed; BRUN records all say May 2001.

In [ ]:
dates = idx['gold'].processing_date.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(dates.index, dates.values, color='0.5')
ax.set_ylabel('GOLD origins'); ax.set_xlabel('processing month')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout(); fig.savefig(OUT_DIR / '47_gold_processing_dates.png', dpi=150); plt.show()
print('Undated GOLD origins:', int(idx['gold'].processing_date.isna().sum()))
print('BRUN processing dates:', idx['brun'].processing_date.value_counts(dropna=False).to_dict())

events = pd.read_csv(EVENT_INDEX)
display(events.preferred_origin_source.value_counts().rename('events with this preferred origin'))
display(events.preferred_magnitude_source.value_counts(dropna=False).rename('events with this preferred magnitude'))
display(events.sources.value_counts().rename('events').to_frame())